## SAiDL Sparsity & Optimization — Part 1 + Part 2

This notebook covers:
- **Part 1:** Compare LoRA, AdaLoRA, and SoRA on **CoLA (GLUE)** with **DeBERTaV3-base**.
- **Part 2:** Replace SoRA proximal update with standard SGD + L1 subgradient, derive and implement both NumPy and PyTorch versions, and compare behavior.

Outputs tracked:
- CoLA validation metric (Matthews Correlation Coefficient)
- Trainable parameters
- Effective rank of adaptation updates
- Training time
- Gate sparsity behavior for proximal vs SGD-subgradient

The cell pattern follows explanation-first style for report readability and reproducibility.

## Explanation for Cell 1: Optional dependency installation

Install these packages if your environment is fresh.

- `transformers`, `datasets`, `evaluate`: model + GLUE pipeline
- `peft`: LoRA and AdaLoRA
- `sentencepiece`, `protobuf`: required for DeBERTaV3 tokenizer backend
- `scikit-learn`: utility metrics support
- `pandas`, `matplotlib`: tables and plots

In [1]:
# Cell 1 — Install dependencies (uncomment if needed)
!pip install -q transformers datasets evaluate peft sentencepiece protobuf tiktoken scikit-learn pandas matplotlib


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Explanation for Cell 2: Imports, accelerator setup, and memory utilities

This cell centralizes reproducibility plus runtime controls:
- explicit CUDA/CPU detection
- mixed-precision policy knobs
- memory reporting and cleanup helpers (`gc` + `torch.cuda.empty_cache`)

These helpers are used between experiments to avoid VRAM accumulation and timing drift.

In [29]:
# Cell 2 — Imports + setup
import os
import re
import gc
import time
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
import evaluate
import transformers
import wandb

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

from peft import (
    LoraConfig,
    AdaLoraConfig,
    get_peft_model,
    TaskType,
)


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
using_cuda = device.type == "cuda"

print("Transformers:", transformers.__version__)
print("Device:", device)
if using_cuda:
    print("GPU:", torch.cuda.get_device_name(0))


def gpu_mem_gb():
    if not using_cuda:
        return {"allocated_gb": 0.0, "reserved_gb": 0.0, "max_allocated_gb": 0.0}
    return {
        "allocated_gb": torch.cuda.memory_allocated() / (1024 ** 3),
        "reserved_gb": torch.cuda.memory_reserved() / (1024 ** 3),
        "max_allocated_gb": torch.cuda.max_memory_allocated() / (1024 ** 3),
    }


def print_gpu_mem(prefix: str = ""):
    if not using_cuda:
        print(f"{prefix}CPU run (no CUDA memory stats)")
        return
    m = gpu_mem_gb()
    print(
        f"{prefix}GPU mem | allocated={m['allocated_gb']:.3f} GB | "
        f"reserved={m['reserved_gb']:.3f} GB | max_alloc={m['max_allocated_gb']:.3f} GB"
    )


def cleanup_cuda(*objs):
    # Drop Python references first
    for obj in objs:
        del obj
    gc.collect()
    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def count_trainable_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def count_total_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


def list_candidate_linear_targets(model: nn.Module, max_show: int = 40):
    names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            names.append(name)
    print(f"Found {len(names)} linear modules. Showing first {max_show}:")
    for n in names[:max_show]:
        print(" -", n)


mcc_metric = evaluate.load("glue", "cola")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return mcc_metric.compute(predictions=preds, references=labels)

Transformers: 5.5.4
Device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## Explanation for Cell 3: Config and dataset/tokenizer preparation (CoLA)

This cell loads CoLA, tokenizes sentence inputs, and builds train/validation splits.

We use MCC (Matthews correlation), the standard CoLA metric.

In [30]:
# Cell 3 — Experiment config + CoLA data prep
@dataclass
class ExpConfig:
    model_name: str = "microsoft/deberta-v3-base"
    max_length: int = 128
    output_root: str = "outputs_saidl_sparsity"

    # trainer params
    epochs: float = 3.0
    train_batch_size: int = 16
    eval_batch_size: int = 32
    lr: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.06
    logging_steps: int = 25
    eval_strategy: str = "epoch"
    save_strategy: str = "no"

    # runtime / accelerator params
    dataloader_num_workers: int = 2
    use_fp16: bool = False
    use_bf16: bool = False

    # LoRA/AdaLoRA base params
    lora_r: int = 8
    lora_alpha: int = 8
    lora_dropout: float = 0.05

    # AdaLoRA-specific params
    adalora_init_r: int = 12
    adalora_target_r: int = 8
    adalora_tinit: int = 100
    adalora_tfinal: int = 500
    adalora_deltaT: int = 10

    # W&B logging
    use_wandb: bool = True
    wandb_project: str = "saidl-sparsity-optimization"
    wandb_entity: str | None = None
    wandb_run_group: str = "part1-part2"

    seed: int = 42


cfg = ExpConfig()
os.makedirs(cfg.output_root, exist_ok=True)
print(cfg)

# Mixed precision policy that is safe on both CPU and CUDA.
if not using_cuda:
    cfg.use_fp16 = False
    cfg.use_bf16 = False
elif cfg.use_bf16:
    cfg.use_fp16 = False

print(f"Runtime policy -> CUDA: {using_cuda}, fp16: {cfg.use_fp16}, bf16: {cfg.use_bf16}")

if cfg.use_wandb:
    os.environ["WANDB_PROJECT"] = cfg.wandb_project
    os.environ["WANDB_RUN_GROUP"] = cfg.wandb_run_group
    if cfg.wandb_entity:
        os.environ["WANDB_ENTITY"] = cfg.wandb_entity
    wandb.login()
    print(f"W&B enabled -> project={cfg.wandb_project}, group={cfg.wandb_run_group}")
else:
    os.environ["WANDB_DISABLED"] = "true"
    print("W&B disabled")

# DeBERTa-v3 tokenization requires sentencepiece (+ protobuf).
try:
    import sentencepiece  # noqa: F401
except Exception as exc:
    raise ImportError(
        "Missing `sentencepiece` for DeBERTaV3 tokenizer. "
        "Run Cell 1 install, restart kernel, then rerun from top."
    ) from exc

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
raw_ds = load_dataset("glue", "cola")


def preprocess(examples):
    return tokenizer(
        examples["sentence"],
        truncation=True,
        padding=False,
        max_length=cfg.max_length,
    )


tok_ds = raw_ds.map(preprocess, batched=True)
tok_ds = tok_ds.rename_column("label", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tok_ds)

ExpConfig(model_name='microsoft/deberta-v3-base', max_length=128, output_root='outputs_saidl_sparsity', epochs=3.0, train_batch_size=16, eval_batch_size=32, lr=2e-05, weight_decay=0.01, warmup_ratio=0.06, logging_steps=25, eval_strategy='epoch', save_strategy='no', dataloader_num_workers=2, use_fp16=False, use_bf16=False, lora_r=8, lora_alpha=8, lora_dropout=0.05, adalora_init_r=12, adalora_target_r=8, adalora_tinit=100, adalora_tfinal=500, adalora_deltaT=10, use_wandb=True, wandb_project='saidl-sparsity-optimization', wandb_entity=None, wandb_run_group='part1-part2', seed=42)
Runtime policy -> CUDA: True, fp16: False, bf16: False
W&B enabled -> project=saidl-sparsity-optimization, group=part1-part2
DatasetDict({
    train: Dataset({
        features: ['sentence', 'labels', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 8551
    })
    validation: Dataset({
        features: ['sentence', 'labels', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
    

## Pre-run sanity check (device + memory)

Run this once before experiments. It:
- prints runtime policy (CUDA + precision)
- prints GPU memory before cleanup
- forces cache cleanup
- prints GPU memory after cleanup

In [31]:
# Pre-run sanity cell — Sparsity notebook
print("=== Runtime policy ===")
print("Device:", device)
print("using_cuda:", using_cuda)
print("fp16:", cfg.use_fp16)
print("bf16:", cfg.use_bf16)

print_gpu_mem("Before cleanup: ")
cleanup_cuda()
print_gpu_mem("After cleanup:  ")
print("Sanity check complete. Safe to start runs.")

=== Runtime policy ===
Device: cuda
using_cuda: True
fp16: False
bf16: False
Before cleanup: GPU mem | allocated=0.000 GB | reserved=0.000 GB | max_alloc=0.000 GB
After cleanup:  GPU mem | allocated=0.000 GB | reserved=0.000 GB | max_alloc=0.000 GB
Sanity check complete. Safe to start runs.


## Deprecated patch cell

No action required here. `build_base_model(...)` is now canonicalized in the main training helper cell.

In [32]:
# Deprecated patch cell (intentionally no-op)
print("No-op: safe loader now lives in the main build_base_model cell.")

No-op: safe loader now lives in the main build_base_model cell.


## Explanation for Cell 4: Common Trainer wrapper (Accelerate + memory metrics)

This function trains any passed model with identical data and optimizer settings.

It also:
- uses Hugging Face Trainer's Accelerate backend implicitly
- enables fp16/bf16 based on config
- tracks peak GPU memory per run

In [33]:
# Cell 4 — Reusable training/evaluation wrapper

def train_and_eval_model(model: nn.Module, run_name: str, cfg: ExpConfig) -> Tuple[Dict[str, float], Trainer]:
    model.to(device)
    # Full FP32: DeBERTa-v3 + PEFT LoRA often produces NaN loss/logits under fp16 autocast.
    model.float()

    out_dir = os.path.join(cfg.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)

    use_bf16 = bool(cfg.use_bf16 and using_cuda)
    use_fp16 = bool(cfg.use_fp16 and using_cuda and not use_bf16)

    args = TrainingArguments(
        output_dir=out_dir,
        overwrite_output_dir=True,
        num_train_epochs=cfg.epochs,
        per_device_train_batch_size=cfg.train_batch_size,
        per_device_eval_batch_size=cfg.eval_batch_size,
        learning_rate=cfg.lr,
        weight_decay=cfg.weight_decay,
        warmup_ratio=cfg.warmup_ratio,
        evaluation_strategy=cfg.eval_strategy,
        save_strategy=cfg.save_strategy,
        logging_steps=cfg.logging_steps,
        dataloader_num_workers=cfg.dataloader_num_workers,
        max_grad_norm=1.0,
        fp16=use_fp16,
        bf16=use_bf16,
        report_to="none",
        seed=cfg.seed,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tok_ds["train"],
        eval_dataset=tok_ds["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainable = count_trainable_params(model)
    total = count_total_params(model)

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    trainer.train()
    train_time_sec = time.time() - start

    eval_out = trainer.evaluate()
    val_mcc = float(eval_out["eval_matthews_correlation"])
    peak_gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if using_cuda else 0.0

    result = {
        "run": run_name,
        "val_mcc": val_mcc,
        "train_time_sec": float(train_time_sec),
        "peak_gpu_mem_gb": peak_gpu_mem_gb,
        "trainable_params": int(trainable),
        "total_params": int(total),
        "trainable_pct": 100.0 * trainable / total,
    }
    return result, trainer


def build_base_model(model_name: str):
    """Canonical model loader: safetensors-first for torch<2.6 compatibility."""
    return AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        use_safetensors=True,
        torch_dtype=torch.float32,
        return_dict=True,
        problem_type="single_label_classification",
    )

## Deprecated hotfix cell

No action required. Keeping this cell only for notebook history; it is now intentionally inactive.

In [34]:
# Deprecated hotfix cell (intentionally no-op)
print("No-op: canonical build_base_model already uses safetensors.")

No-op: canonical build_base_model already uses safetensors.


## Compatibility patch: TrainingArguments API drift

This cell redefines `train_and_eval_model(...)` to dynamically filter `TrainingArguments` kwargs based on the installed `transformers` version (e.g., handles missing `overwrite_output_dir`).

In [35]:
# Compatibility patch cell — run before Cell 5
import inspect

def train_and_eval_model(model: nn.Module, run_name: str, cfg: ExpConfig) -> Tuple[Dict[str, float], Trainer]:
    model.to(device)
    model.float()

    out_dir = os.path.join(cfg.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)

    use_bf16 = bool(getattr(cfg, "use_bf16", False) and using_cuda)
    use_fp16 = bool(getattr(cfg, "use_fp16", False) and using_cuda and not use_bf16)

    # Stable default: FP32 unless config explicitly requests bf16/fp16.
    maybe_kwargs = {
        "output_dir": out_dir,
        "num_train_epochs": cfg.epochs,
        "per_device_train_batch_size": cfg.train_batch_size,
        "per_device_eval_batch_size": cfg.eval_batch_size,
        "learning_rate": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "warmup_ratio": cfg.warmup_ratio,
        "evaluation_strategy": cfg.eval_strategy,
        "eval_strategy": cfg.eval_strategy,
        "save_strategy": cfg.save_strategy,
        "logging_steps": cfg.logging_steps,
        "dataloader_num_workers": cfg.dataloader_num_workers,
        "max_grad_norm": 1.0,
        "fp16": use_fp16,
        "bf16": use_bf16,
        "disable_tqdm": True,
        "report_to": (["wandb"] if getattr(cfg, "use_wandb", False) else "none"),
        "run_name": run_name,
        "seed": cfg.seed,
    }
    valid = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = TrainingArguments(**{k: v for k, v in maybe_kwargs.items() if k in valid})

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tok_ds["train"],
        "eval_dataset": tok_ds["validation"],
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }
    trainer_valid = set(inspect.signature(Trainer.__init__).parameters)
    if "processing_class" in trainer_valid:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_valid:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainable = count_trainable_params(model)
    total = count_total_params(model)

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    trainer.train()
    train_time_sec = time.time() - start

    eval_out = trainer.evaluate()
    val_mcc = float(eval_out["eval_matthews_correlation"])
    peak_gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if using_cuda else 0.0

    result = {
        "run": run_name,
        "val_mcc": val_mcc,
        "train_time_sec": float(train_time_sec),
        "peak_gpu_mem_gb": peak_gpu_mem_gb,
        "trainable_params": int(trainable),
        "total_params": int(total),
        "trainable_pct": 100.0 * trainable / total,
    }

    if getattr(cfg, "use_wandb", False) and wandb.run is not None:
        wandb.log({
            "final/val_mcc": result["val_mcc"],
            "system/train_time_sec": result["train_time_sec"],
            "system/peak_gpu_mem_gb": result["peak_gpu_mem_gb"],
            "params/trainable": result["trainable_params"],
            "params/total": result["total_params"],
            "params/trainable_pct": result["trainable_pct"],
        })

    return result, trainer

print("Patched train_and_eval_model for current TrainingArguments signature")

Patched train_and_eval_model for current TrainingArguments signature


## Compatibility patch: SoRA W&B logging

This redefines `train_sora_model(...)` to include optional W&B logging for the manual training loop (epoch loss + final metrics).

In [36]:
# SoRA W&B patch cell — run before SoRA training cell

def train_sora_model(cfg: ExpConfig, target_suffixes: List[str], l1_lambda: float = 1e-4):
    model = build_base_model(cfg.model_name)
    model = apply_sora_to_model(model, target_suffixes, rank=cfg.lora_r, alpha=cfg.lora_alpha)
    model.to(device)
    model.float()

    sora_run = None
    if getattr(cfg, "use_wandb", False):
        sora_run = wandb.init(
            project=cfg.wandb_project,
            entity=cfg.wandb_entity,
            group=cfg.wandb_run_group,
            name="sora_prox",
            reinit="finish_previous",
            config=asdict(cfg),
            tags=["sora", "cola", "part1"],
        )

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    model.train()
    global_step = 0
    for epoch in range(int(math.ceil(cfg.epochs))):
        epoch_losses = []
        for batch in train_loader:
            global_step += 1
            batch = {k: v.to(device) for k, v in batch.items()}

            opt.zero_grad()
            out = model(**batch)
            task_loss = out.loss

            l1_term = 0.0
            for m in model.modules():
                if isinstance(m, SoraLinear):
                    l1_term = l1_term + m.g.abs().sum()

            loss = task_loss + l1_lambda * l1_term
            loss.backward()
            opt.step()
            proximal_step_on_g(model, lr=cfg.lr, lam=l1_lambda)
            epoch_losses.append(float(loss.item()))

        if sora_run is not None and len(epoch_losses) > 0:
            wandb.log({"train/epoch_loss": float(np.mean(epoch_losses)), "epoch": epoch + 1}, step=global_step)

    train_time_sec = time.time() - start
    val_mcc = eval_mcc_manual(model, val_loader)
    peak_gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if using_cuda else 0.0

    rank_stats = get_sora_effective_rank_stats(model)

    result = {
        "run": "sora_prox",
        "val_mcc": val_mcc,
        "train_time_sec": float(train_time_sec),
        "peak_gpu_mem_gb": peak_gpu_mem_gb,
        "trainable_params": int(count_trainable_params(model)),
        "total_params": int(count_total_params(model)),
        "trainable_pct": 100.0 * count_trainable_params(model) / count_total_params(model),
        **rank_stats,
    }

    if sora_run is not None:
        wandb.log({
            "final/val_mcc": result["val_mcc"],
            "system/train_time_sec": result["train_time_sec"],
            "system/peak_gpu_mem_gb": result["peak_gpu_mem_gb"],
            "rank/avg_effective_rank": result["avg_effective_rank"],
            "rank/max_effective_rank": result["max_effective_rank"],
            "rank/gate_sparsity": result["gate_sparsity"],
        })
        wandb.finish()

    return model, result

print("Patched train_sora_model with optional W&B logging")

Patched train_sora_model with optional W&B logging


## Runtime patch: disable notebook callback crash

This redefines `train_and_eval_model(...)` with `disable_tqdm=True` and version-safe Trainer kwargs to avoid `on_train_begin must be called before on_evaluate` in this Transformers notebook environment.

In [37]:
# Notebook-callback stability patch — run before Cell 5
import inspect

def train_and_eval_model(model: nn.Module, run_name: str, cfg: ExpConfig) -> Tuple[Dict[str, float], Trainer]:
    model.to(device)
    model.float()

    out_dir = os.path.join(cfg.output_root, run_name)
    os.makedirs(out_dir, exist_ok=True)

    use_bf16 = bool(getattr(cfg, "use_bf16", False) and using_cuda)
    use_fp16 = bool(getattr(cfg, "use_fp16", False) and using_cuda and not use_bf16)

    maybe_kwargs = {
        "output_dir": out_dir,
        "num_train_epochs": cfg.epochs,
        "per_device_train_batch_size": cfg.train_batch_size,
        "per_device_eval_batch_size": cfg.eval_batch_size,
        "learning_rate": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "warmup_ratio": cfg.warmup_ratio,
        "evaluation_strategy": cfg.eval_strategy,
        "eval_strategy": cfg.eval_strategy,
        "save_strategy": cfg.save_strategy,
        "logging_steps": cfg.logging_steps,
        "dataloader_num_workers": cfg.dataloader_num_workers,
        "max_grad_norm": 1.0,
        "fp16": use_fp16,
        "bf16": use_bf16,
        "disable_tqdm": True,
        "report_to": (["wandb"] if getattr(cfg, "use_wandb", False) else "none"),
        "run_name": run_name,
        "seed": cfg.seed,
    }
    valid = set(inspect.signature(TrainingArguments.__init__).parameters)
    args = TrainingArguments(**{k: v for k, v in maybe_kwargs.items() if k in valid})

    trainer_kwargs = {
        "model": model,
        "args": args,
        "train_dataset": tok_ds["train"],
        "eval_dataset": tok_ds["validation"],
        "data_collator": data_collator,
        "compute_metrics": compute_metrics,
    }
    trainer_valid = set(inspect.signature(Trainer.__init__).parameters)
    if "processing_class" in trainer_valid:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_valid:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = Trainer(**trainer_kwargs)

    trainable = count_trainable_params(model)
    total = count_total_params(model)

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    trainer.train()
    train_time_sec = time.time() - start

    eval_out = trainer.evaluate()
    val_mcc = float(eval_out["eval_matthews_correlation"])
    peak_gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if using_cuda else 0.0

    result = {
        "run": run_name,
        "val_mcc": val_mcc,
        "train_time_sec": float(train_time_sec),
        "peak_gpu_mem_gb": peak_gpu_mem_gb,
        "trainable_params": int(trainable),
        "total_params": int(total),
        "trainable_pct": 100.0 * trainable / total,
    }

    if getattr(cfg, "use_wandb", False) and wandb.run is not None:
        wandb.log({
            "final/val_mcc": result["val_mcc"],
            "system/train_time_sec": result["train_time_sec"],
            "system/peak_gpu_mem_gb": result["peak_gpu_mem_gb"],
            "params/trainable": result["trainable_params"],
            "params/total": result["total_params"],
            "params/trainable_pct": result["trainable_pct"],
        })

    return result, trainer

print("Patched train_and_eval_model: stable notebook callbacks + API compatibility")

Patched train_and_eval_model: stable notebook callbacks + API compatibility


## Runtime patch: robust PEFT rank stats

Use this before the LoRA/AdaLoRA cell to avoid SVD crashes when adapter matrices contain NaN/Inf values.

In [38]:
# Robust rank-stats patch cell — run before Cell 5

def peft_effective_rank_stats(peft_model: nn.Module, tol: float = 1e-5):
    ranks = []
    skipped_non_finite = 0
    svd_failures = 0

    for module in peft_model.modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            for key in module.lora_A.keys():
                A_mod = module.lora_A[key]
                B_mod = module.lora_B[key]
                A = A_mod.weight.detach().float().cpu()
                B = B_mod.weight.detach().float().cpu()
                dw = B @ A

                if not torch.isfinite(dw).all():
                    skipped_non_finite += 1
                    dw = torch.nan_to_num(dw, nan=0.0, posinf=0.0, neginf=0.0)

                try:
                    s = torch.linalg.svdvals(dw)
                    rk = int((s > tol).sum().item())
                    ranks.append(rk)
                except RuntimeError:
                    svd_failures += 1

    out = {
        "avg_effective_rank": float(np.mean(ranks)) if ranks else 0.0,
        "max_effective_rank": int(np.max(ranks)) if ranks else 0,
        "rank_modules_used": int(len(ranks)),
        "rank_non_finite_sanitized": int(skipped_non_finite),
        "rank_svd_failures": int(svd_failures),
    }

    if skipped_non_finite > 0 or svd_failures > 0:
        print(
            f"[rank-stats] sanitized_non_finite={skipped_non_finite}, "
            f"svd_failures={svd_failures}, used={len(ranks)}"
        )

    return out

print("Patched peft_effective_rank_stats for non-finite-safe SVD")

Patched peft_effective_rank_stats for non-finite-safe SVD


## Stability override (run before Cell 5)

This cell applies conservative settings and safe helper overrides to prevent NaN divergence:
- lower LR
- disable mixed precision
- attention-only LoRA targets
- non-finite-safe rank SVD

In [39]:
# Stability override cell — run this before Cell 5

# Conservative runtime/training defaults
cfg.lr = 2e-5
cfg.use_fp16 = False
cfg.use_bf16 = False
cfg.lora_alpha = 8


def guess_target_modules(model: nn.Module) -> List[str]:
    # Attention projections only (exclude broad dense adapters for stability)
    candidate_suffixes = ["query_proj", "key_proj", "value_proj"]
    found_suffixes = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            for s in candidate_suffixes:
                if name.endswith(s):
                    found_suffixes.add(s)
    if not found_suffixes:
        candidate_suffixes = ["query", "key", "value"]
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                for s in candidate_suffixes:
                    if name.endswith(s):
                        found_suffixes.add(s)
    return sorted(found_suffixes)


def peft_effective_rank_stats(peft_model: nn.Module, tol: float = 1e-5):
    ranks = []
    skipped_non_finite = 0
    svd_failures = 0

    for module in peft_model.modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            for key in module.lora_A.keys():
                A_mod = module.lora_A[key]
                B_mod = module.lora_B[key]
                A = A_mod.weight.detach().float().cpu()
                B = B_mod.weight.detach().float().cpu()
                dw = B @ A

                if not torch.isfinite(dw).all():
                    skipped_non_finite += 1
                    dw = torch.nan_to_num(dw, nan=0.0, posinf=0.0, neginf=0.0)

                try:
                    s = torch.linalg.svdvals(dw)
                    rk = int((s > tol).sum().item())
                    ranks.append(rk)
                except RuntimeError:
                    svd_failures += 1

    out = {
        "avg_effective_rank": float(np.mean(ranks)) if ranks else 0.0,
        "max_effective_rank": int(np.max(ranks)) if ranks else 0,
        "rank_modules_used": int(len(ranks)),
        "rank_non_finite_sanitized": int(skipped_non_finite),
        "rank_svd_failures": int(svd_failures),
    }

    if skipped_non_finite > 0 or svd_failures > 0:
        print(
            f"[rank-stats] sanitized_non_finite={skipped_non_finite}, "
            f"svd_failures={svd_failures}, used={len(ranks)}"
        )

    return out

print("Applied stability overrides: lr=2e-5, fp16/bf16 off, lora_alpha=8, attention-only targets")

Applied stability overrides: lr=2e-5, fp16/bf16 off, lora_alpha=8, attention-only targets


## Explanation for Cell 5: Part 1 — LoRA and AdaLoRA runs with explicit cleanup

This cell runs LoRA and AdaLoRA sequentially, computes effective-rank stats immediately, then frees Trainer/model objects to prevent residual VRAM occupancy before the next run.

In [40]:
# Cell 5 — LoRA + AdaLoRA experiments (Part 1)

# Good default targets for DeBERTa-style attention projections.
# Keep this attention-only to avoid unstable broad adapter injection.
def guess_target_modules(model: nn.Module) -> List[str]:
    candidate_suffixes = ["query_proj", "key_proj", "value_proj"]
    found_suffixes = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            for s in candidate_suffixes:
                if name.endswith(s):
                    found_suffixes.add(s)
    if not found_suffixes:
        # fallback to common BERT naming
        candidate_suffixes = ["query", "key", "value"]
        for name, module in model.named_modules():
            if isinstance(module, nn.Linear):
                for s in candidate_suffixes:
                    if name.endswith(s):
                        found_suffixes.add(s)
    return sorted(found_suffixes)


def peft_effective_rank_stats(peft_model: nn.Module, tol: float = 1e-5):
    ranks = []
    skipped_non_finite = 0
    svd_failures = 0

    for module in peft_model.modules():
        if hasattr(module, "lora_A") and hasattr(module, "lora_B"):
            # lora_A/lora_B are often ModuleDict keyed by adapter name
            for key in module.lora_A.keys():
                A_mod = module.lora_A[key]
                B_mod = module.lora_B[key]
                A = A_mod.weight.detach().float().cpu()
                B = B_mod.weight.detach().float().cpu()
                dw = B @ A

                if not torch.isfinite(dw).all():
                    skipped_non_finite += 1
                    dw = torch.nan_to_num(dw, nan=0.0, posinf=0.0, neginf=0.0)

                try:
                    s = torch.linalg.svdvals(dw)
                    rk = int((s > tol).sum().item())
                    ranks.append(rk)
                except RuntimeError:
                    svd_failures += 1

    return {
        "avg_effective_rank": float(np.mean(ranks)) if ranks else 0.0,
        "max_effective_rank": int(np.max(ranks)) if ranks else 0,
        "rank_modules_used": int(len(ranks)),
        "rank_non_finite_sanitized": int(skipped_non_finite),
        "rank_svd_failures": int(svd_failures),
    }


base_model_for_targets = build_base_model(cfg.model_name)
target_modules = guess_target_modules(base_model_for_targets)
print("Target module suffixes:", target_modules)
if len(target_modules) == 0:
    print("No target suffixes auto-detected. Inspect modules with list_candidate_linear_targets(base_model_for_targets).")
cleanup_cuda(base_model_for_targets)

results = []

# LoRA
print_gpu_mem("Before LoRA: ")
base_lora = build_base_model(cfg.model_name)
lora_conf = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=target_modules,
    bias="none",
    total_step=total_steps,
)
model_lora = get_peft_model(base_lora, lora_conf)
lora_result, lora_trainer = train_and_eval_model(model_lora, "lora", cfg)
lora_result.update(peft_effective_rank_stats(model_lora))
results.append(lora_result)
cleanup_cuda(lora_trainer, model_lora, base_lora)
print_gpu_mem("After LoRA cleanup: ")

# AdaLoRA
print_gpu_mem("Before AdaLoRA: ")
base_adalora = build_base_model(cfg.model_name)
total_steps = math.ceil(len(tok_ds["train"]) / cfg.train_batch_size) * int(math.ceil(cfg.epochs))
adalora_conf = AdaLoraConfig(
    task_type=TaskType.SEQ_CLS,
    init_r=cfg.adalora_init_r,
    target_r=cfg.adalora_target_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    target_modules=target_modules,
    tinit=cfg.adalora_tinit,
    tfinal=cfg.adalora_tfinal,
    deltaT=cfg.adalora_deltaT,
    bias="none",
    total_step=total_steps,
)
model_adalora = get_peft_model(base_adalora, adalora_conf)
adalora_result, adalora_trainer = train_and_eval_model(model_adalora, "adalora", cfg)
adalora_result.update(peft_effective_rank_stats(model_adalora))
results.append(adalora_result)
cleanup_cuda(adalora_trainer, model_adalora, base_adalora)
print_gpu_mem("After AdaLoRA cleanup: ")

pd.DataFrame(results)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1924.71it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.w

Target module suffixes: ['key_proj', 'query_proj', 'value_proj']
Before LoRA: GPU mem | allocated=0.000 GB | reserved=0.000 GB | max_alloc=0.000 GB


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 1521.54it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.w

NameError: name 'total_steps' is not defined

## Explanation for Cell 6: SoRA module + proximal update

This is a lightweight SoRA-like implementation:
- Freeze original linear weight
- Add low-rank update \(\Delta W = (B \odot g)A\)
- Apply **proximal soft-thresholding** on gate vector `g` after gradient step to induce sparsity

In [ ]:
# Cell 6 — SoRA implementation + proximal step
class SoraLinear(nn.Module):
    def __init__(self, base_linear: nn.Linear, rank: int = 8, alpha: float = 16.0, gate_init: float = 1.0):
        super().__init__()
        self.in_features = base_linear.in_features
        self.out_features = base_linear.out_features
        self.rank = rank
        self.scale = alpha / rank

        # Frozen base
        self.base = base_linear
        self.base.weight.requires_grad = False
        if self.base.bias is not None:
            self.base.bias.requires_grad = False

        # Trainable low-rank path
        self.A = nn.Parameter(torch.zeros(rank, self.in_features))
        self.B = nn.Parameter(torch.zeros(self.out_features, rank))
        self.g = nn.Parameter(torch.full((rank,), gate_init))

        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

    def delta_weight(self):
        gated_B = self.B * self.g.unsqueeze(0)
        return (gated_B @ self.A) * self.scale

    def forward(self, x):
        base_out = self.base(x)
        dw = self.delta_weight()
        lora_out = F.linear(x, dw, bias=None)
        return base_out + lora_out


def get_module_by_name(root: nn.Module, name: str):
    curr = root
    for p in name.split('.'):
        curr = getattr(curr, p)
    return curr


def set_module_by_name(root: nn.Module, name: str, module: nn.Module):
    parts = name.split('.')
    parent = root
    for p in parts[:-1]:
        parent = getattr(parent, p)
    setattr(parent, parts[-1], module)


def apply_sora_to_model(model: nn.Module, target_suffixes: List[str], rank: int = 8, alpha: float = 16.0):
    replace_names = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear) and any(name.endswith(s) for s in target_suffixes):
            replace_names.append(name)

    for name in replace_names:
        old = get_module_by_name(model, name)
        new = SoraLinear(old, rank=rank, alpha=alpha)
        set_module_by_name(model, name, new)

    # Freeze all params first
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze SoRA params and classifier head
    for m in model.modules():
        if isinstance(m, SoraLinear):
            m.A.requires_grad = True
            m.B.requires_grad = True
            m.g.requires_grad = True

    for n, p in model.named_parameters():
        if "classifier" in n:
            p.requires_grad = True

    return model


@torch.no_grad()
def proximal_step_on_g(model: nn.Module, lr: float, lam: float):
    # Soft-threshold: g <- sign(g) * max(|g| - lr*lam, 0)
    thr = lr * lam
    for m in model.modules():
        if isinstance(m, SoraLinear):
            g = m.g.data
            m.g.data = torch.sign(g) * torch.clamp(torch.abs(g) - thr, min=0.0)


def get_sora_effective_rank_stats(model: nn.Module, tol: float = 1e-5):
    ranks = []
    nonzero_gate_counts = []
    total_gates = []

    for m in model.modules():
        if isinstance(m, SoraLinear):
            dw = m.delta_weight().detach().float().cpu()
            s = torch.linalg.svdvals(dw)
            rk = int((s > tol).sum().item())
            ranks.append(rk)

            g = m.g.detach()
            nz = int((g.abs() > 1e-8).sum().item())
            nonzero_gate_counts.append(nz)
            total_gates.append(g.numel())

    out = {
        "avg_effective_rank": float(np.mean(ranks)) if ranks else 0.0,
        "max_effective_rank": int(np.max(ranks)) if ranks else 0,
        "avg_nonzero_gates": float(np.mean(nonzero_gate_counts)) if nonzero_gate_counts else 0.0,
        "gate_sparsity": 1.0 - (sum(nonzero_gate_counts) / max(1, sum(total_gates))),
    }
    return out

## Explanation for Cell 7: SoRA training loop (Part 1) with CUDA cleanup

We use a simple manual loop because we need explicit proximal update control for gates after optimizer updates.

Loss optimized:
\[
\mathcal{L}(\Delta)=\mathcal{L}_0(\Delta)+\lambda\sum_k\|g^{(k)}\|_1
\]
with proximal step on `g` to enforce sparsity.

This cell also resets peak-memory stats before training and clears GPU memory after collecting metrics.

In [ ]:
# Cell 7 — SoRA train/eval
from torch.utils.data import DataLoader


def to_torch_dataset(split_ds):
    cols = ["input_ids", "attention_mask", "labels"]
    return split_ds.remove_columns([c for c in split_ds.column_names if c not in cols]).with_format("torch")


train_ds_t = to_torch_dataset(tok_ds["train"])
val_ds_t = to_torch_dataset(tok_ds["validation"])

# Use dynamic padding for variable-length tokenized sentences.
train_loader = DataLoader(
    train_ds_t,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    collate_fn=data_collator,
)
val_loader = DataLoader(
    val_ds_t,
    batch_size=cfg.eval_batch_size,
    shuffle=False,
    collate_fn=data_collator,
)


def eval_mcc_manual(model: nn.Module, val_loader: DataLoader) -> float:
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            preds = out.logits.argmax(dim=-1)
            all_preds.append(preds.cpu().numpy())
            all_labels.append(batch["labels"].cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return float(mcc_metric.compute(predictions=preds, references=labels)["matthews_correlation"])


def train_sora_model(cfg: ExpConfig, target_suffixes: List[str], l1_lambda: float = 1e-4):
    model = build_base_model(cfg.model_name)
    model = apply_sora_to_model(model, target_suffixes, rank=cfg.lora_r, alpha=cfg.lora_alpha)
    model.to(device)
    model.float()

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr, weight_decay=cfg.weight_decay)

    if using_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    start = time.time()
    model.train()
    for epoch in range(int(math.ceil(cfg.epochs))):
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            opt.zero_grad()
            out = model(**batch)
            task_loss = out.loss

            # Smooth L1 penalty term (value only); sparsity is enforced via proximal step.
            l1_term = 0.0
            for m in model.modules():
                if isinstance(m, SoraLinear):
                    l1_term = l1_term + m.g.abs().sum()

            loss = task_loss + l1_lambda * l1_term
            loss.backward()
            opt.step()

            # Proximal operator on gates
            proximal_step_on_g(model, lr=cfg.lr, lam=l1_lambda)

    train_time_sec = time.time() - start
    val_mcc = eval_mcc_manual(model, val_loader)
    peak_gpu_mem_gb = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if using_cuda else 0.0

    rank_stats = get_sora_effective_rank_stats(model)

    result = {
        "run": "sora_prox",
        "val_mcc": val_mcc,
        "train_time_sec": float(train_time_sec),
        "peak_gpu_mem_gb": peak_gpu_mem_gb,
        "trainable_params": int(count_trainable_params(model)),
        "total_params": int(count_total_params(model)),
        "trainable_pct": 100.0 * count_trainable_params(model) / count_total_params(model),
        **rank_stats,
    }
    return model, result


print_gpu_mem("Before SoRA: ")
model_sora, sora_result = train_sora_model(cfg, target_modules, l1_lambda=1e-4)
results.append(sora_result)
cleanup_cuda(model_sora)
print_gpu_mem("After SoRA cleanup: ")
pd.DataFrame(results)

Before SoRA: GPU mem | allocated=0.382 GB | reserved=1.543 GB | max_alloc=1.403 GB


Loading weights: 100%|██████████| 198/198 [00:00<00:00, 6194.69it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier.b

RuntimeError: stack expects each tensor to be equal size, but got [10] at entry 0 and [15] at entry 1

## Explanation for Cell 8: Part 1 summary table

Effective-rank values are already computed during each run, before cleanup. This cell just assembles and sorts the final comparison table.

In [ ]:
# Cell 8 — Final Part 1 table
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("val_mcc", ascending=False).reset_index(drop=True)
results_df

## Explanation for Cell 9: Part 2 derivation (SGD subgradient vs proximal)

For a gate coordinate \(g_i\):

- Objective: \(f(g) + \lambda\|g\|_1\)
- **SGD + subgradient** update:
\[
g_i^{t+1}=g_i^t-\eta\left(\frac{\partial f}{\partial g_i}+\lambda s_i\right),\quad s_i\in\partial|g_i|
\]
with
\[
\partial |g_i|=
\begin{cases}
\{+1\}, & g_i>0\\
[-1,1], & g_i=0\\
\{-1\}, & g_i<0
\end{cases}
\]

- **Proximal gradient** update:
\[
\nu_i=g_i^t-\eta\frac{\partial f}{\partial g_i},\qquad
g_i^{t+1}=\operatorname{sign}(\nu_i)\max(|\nu_i|-\eta\lambda,0)
\]

Key difference: proximal step performs exact soft-thresholding and creates true zeros robustly; SGD-subgradient usually shrinks but does not enforce exact sparsity as aggressively.

In [ ]:
# Cell 9 — NumPy and PyTorch implementations for Part 2
import matplotlib.pyplot as plt


def l1_subgradient(x: np.ndarray):
    # choose 0 at exact zero (valid subgradient choice)
    g = np.sign(x)
    g[np.isclose(x, 0.0)] = 0.0
    return g


def sgd_subgrad_update_numpy(g, grad_f, lr, lam):
    return g - lr * (grad_f + lam * l1_subgradient(g))


def proximal_update_numpy(g, grad_f, lr, lam):
    nu = g - lr * grad_f
    return np.sign(nu) * np.maximum(np.abs(nu) - lr * lam, 0.0)


def run_gate_dynamics_numpy(steps=200, dim=64, lr=1e-2, lam=1e-2, seed=0):
    rng = np.random.default_rng(seed)
    target = rng.normal(0, 0.1, size=(dim,))

    g_sgd = rng.normal(0, 0.3, size=(dim,))
    g_prox = g_sgd.copy()

    sgd_nonzero, prox_nonzero = [], []
    sgd_dist, prox_dist = [], []

    for _ in range(steps):
        grad_sgd = (g_sgd - target)
        grad_prox = (g_prox - target)

        g_sgd = sgd_subgrad_update_numpy(g_sgd, grad_sgd, lr, lam)
        g_prox = proximal_update_numpy(g_prox, grad_prox, lr, lam)

        sgd_nonzero.append(np.sum(np.abs(g_sgd) > 1e-8))
        prox_nonzero.append(np.sum(np.abs(g_prox) > 1e-8))
        sgd_dist.append(np.linalg.norm(g_sgd - target))
        prox_dist.append(np.linalg.norm(g_prox - target))

    return {
        "sgd_nonzero": np.array(sgd_nonzero),
        "prox_nonzero": np.array(prox_nonzero),
        "sgd_dist": np.array(sgd_dist),
        "prox_dist": np.array(prox_dist),
    }


def sgd_subgrad_update_torch(g, grad_f, lr, lam):
    subgrad = torch.sign(g)
    subgrad = torch.where(torch.isclose(g, torch.zeros_like(g)), torch.zeros_like(g), subgrad)
    return g - lr * (grad_f + lam * subgrad)


def proximal_update_torch(g, grad_f, lr, lam):
    nu = g - lr * grad_f
    return torch.sign(nu) * torch.clamp(torch.abs(nu) - lr * lam, min=0.0)


def run_gate_dynamics_torch(steps=200, dim=64, lr=1e-2, lam=1e-2, seed=0):
    torch.manual_seed(seed)
    target = torch.randn(dim) * 0.1
    g0 = torch.randn(dim) * 0.3
    g_sgd = g0.clone()
    g_prox = g0.clone()

    sgd_nonzero, prox_nonzero = [], []
    sgd_dist, prox_dist = [], []

    for _ in range(steps):
        grad_sgd = g_sgd - target
        grad_prox = g_prox - target

        g_sgd = sgd_subgrad_update_torch(g_sgd, grad_sgd, lr, lam)
        g_prox = proximal_update_torch(g_prox, grad_prox, lr, lam)

        sgd_nonzero.append(int((g_sgd.abs() > 1e-8).sum().item()))
        prox_nonzero.append(int((g_prox.abs() > 1e-8).sum().item()))
        sgd_dist.append(float(torch.norm(g_sgd - target).item()))
        prox_dist.append(float(torch.norm(g_prox - target).item()))

    return {
        "sgd_nonzero": np.array(sgd_nonzero),
        "prox_nonzero": np.array(prox_nonzero),
        "sgd_dist": np.array(sgd_dist),
        "prox_dist": np.array(prox_dist),
    }


np_res = run_gate_dynamics_numpy()
torch_res = run_gate_dynamics_torch()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(np_res["sgd_nonzero"], label="SGD-subgrad (NumPy)")
axes[0].plot(np_res["prox_nonzero"], label="Proximal (NumPy)")
axes[0].set_title("Non-zero gate count")
axes[0].set_xlabel("step")
axes[0].set_ylabel("count")
axes[0].legend()

axes[1].plot(np_res["sgd_dist"], label="SGD-subgrad (NumPy)")
axes[1].plot(np_res["prox_dist"], label="Proximal (NumPy)")
axes[1].set_title("Distance to target gate vector")
axes[1].set_xlabel("step")
axes[1].set_ylabel("L2 distance")
axes[1].legend()
plt.show()

print("Final non-zero gates (NumPy): SGD=", np_res["sgd_nonzero"][-1], ", Prox=", np_res["prox_nonzero"][-1])
print("Final non-zero gates (Torch): SGD=", torch_res["sgd_nonzero"][-1], ", Prox=", torch_res["prox_nonzero"][-1])

## Explanation for Cell 10: Theoretical comparison note for report

- **Where they differ:** proximal applies the exact proximal map of the non-smooth \(\ell_1\) term after a smooth gradient step; SGD-subgradient merges both terms into one noisy first-order step.
- **At zero:** subgradient is set-valued \([-1,1]\), so SGD behavior depends on the selected subgradient convention near zero.
- **Implication:** subgradient choice can change the local trajectory near zero, but does not remove the fundamental gap: proximal update has explicit thresholding and stronger sparsity induction.
- **Practical expectation:** SoRA with proximal gating should produce lower effective rank (higher sparsity) at similar or better task performance-efficiency tradeoff.